# 🧬 Módulo 4: Modelado de Proteínas y Docking Molecular
## Actividad 4.5: Fundamentos de Docking Molecular

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_04_modelado_proteinas_docking/05_docking_fundamentos.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender la teoría del docking molecular
- Conocer métodos de sampling conformacional
- Entender funciones de scoring
- Diferenciar docking rígido vs flexible
- Configurar parámetros de docking
- Interpretar resultados preliminares

---

## 📚 Introducción

El docking molecular es una técnica computacional para predecir la conformación y orientación de un ligando en el sitio activo de una proteína.

---

In [ ]:
# Instalación de dependencias
!pip install numpy pandas matplotlib scipy rdkit-pypi 2>/dev/null || pip install numpy pandas matplotlib scipy rdkit
!pip install py3Dmol
print("✓ Dependencias instaladas")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas correctamente")

## 📚 Introducción al Docking Molecular

El **docking molecular** es una técnica computacional que predice:
1. La **conformación** de un ligando dentro del sitio de unión de una proteína
2. La **afinidad de unión** estimada entre ligando y receptor

### Aplicaciones principales

- 💊 Diseño racional de fármacos
- 🔬 Predicción de mecanismos de acción
- 🧬 Estudio de interacciones moleculares
- 📊 Cribado virtual de librerías de compuestos

### Hipótesis fundamentales

El docking molecular se basa en dos modelos complementarios:

| Modelo | Descripción |
|--------|-------------|
| **Llave-cerradura** (Fisher, 1894) | Proteína y ligando son rígidos y complementarios |
| **Inducción de ajuste** (Koshland, 1958) | La proteína cambia conformación al unirse al ligando |
| **Selección conformacional** | El ligando selecciona una conformación pre-existente |

## 1. Energética de la Unión Ligando-Receptor

La energía libre de Gibbs de unión ($\Delta G_{bind}$) determina la afinidad:

$$\Delta G_{bind} = \Delta H_{bind} - T\Delta S_{bind}$$

$$\Delta G_{bind} = RT \ln K_d$$

Donde $K_d$ es la constante de disociación. Una $\Delta G_{bind}$ más negativa indica **mayor afinidad**.

### Componentes energéticos

$$\Delta G_{bind} = \Delta G_{vdW} + \Delta G_{elec} + \Delta G_{H-bond} + \Delta G_{desolvat} + \Delta G_{conf}$$

| Término | Descripción |
|---------|-------------|
| $\Delta G_{vdW}$ | Van der Waals (favorable si geometría complementaria) |
| $\Delta G_{elec}$ | Electrostático (interacciones carga-carga, dipolo) |
| $\Delta G_{H-bond}$ | Puentes de hidrógeno |
| $\Delta G_{desolvat}$ | Desolvatación (pérdida de agua) |
| $\Delta G_{conf}$ | Penalización conformacional del ligando |

In [ ]:
def kd_to_dg(kd_M, T=298.15):
    """Convierte Kd (M) a ΔG (kcal/mol)."""
    R = 1.987e-3  # kcal/(mol·K)
    return R * T * np.log(kd_M)

def dg_to_kd(dg_kcal, T=298.15):
    """Convierte ΔG (kcal/mol) a Kd (M)."""
    R = 1.987e-3
    return np.exp(dg_kcal / (R * T))

# Tabla de conversión ΔG ↔ Kd a 25°C
print(f"{'ΔG (kcal/mol)':>18} | {'Kd':>15} | {'Potencia'}") 
print("-"*50)
for kd, label in [(1e-4,'100 μM'), (1e-5,'10 μM'), (1e-6,'1 μM'),
                   (1e-7,'100 nM'), (1e-8,'10 nM'), (1e-9,'1 nM'),
                   (1e-10,'100 pM')]:
    dg = kd_to_dg(kd)
    print(f"{dg:>18.2f} | {label:>15} | {'✓ Moderado' if dg > -9 else '✓✓ Bueno' if dg > -12 else '✓✓✓ Excelente'}")

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

kd_range = np.logspace(-10, -3, 100)
dg_range = kd_to_dg(kd_range)

ax1.semilogx(kd_range, dg_range, 'b-', linewidth=2)
ax1.axhline(-6,  color='orange', linestyle='--', alpha=0.7, label='Límite débil (~1 μM)')
ax1.axhline(-9,  color='green',  linestyle='--', alpha=0.7, label='Buen fármaco (~1 nM)')
ax1.axhline(-12, color='red',    linestyle='--', alpha=0.7, label='Excelente (~1 pM)')
ax1.set_xlabel('Kd (M)')
ax1.set_ylabel('ΔG (kcal/mol)')
ax1.set_title('Relación ΔG — Kd')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Componentes energéticos típicos
componentes = ['vdW', 'Electrostático', 'H-bonds', 'Desolvatación', 'Conformacional']
valores_tipicos = [-5.2, -2.1, -1.8, 2.3, 1.5]
colores = ['steelblue' if v < 0 else 'salmon' for v in valores_tipicos]
ax2.barh(componentes, valores_tipicos, color=colores, edgecolor='white')
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('Contribución (kcal/mol)')
ax2.set_title('Contribuciones Típicas al ΔG')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 2. Funciones de Scoring

Las **funciones de scoring** estiman $\Delta G_{bind}$ a partir de la pose del ligando. Existen tres tipos principales:

### 2.1 Funciones de Fuerza
Basadas en campos de fuerza (Lennard-Jones, electrostática):
$$E_{score} = \sum_{pairs} \left[ \frac{A_{ij}}{r_{ij}^{12}} - \frac{B_{ij}}{r_{ij}^6} + \frac{q_i q_j}{4\pi\epsilon r_{ij}} \right]$$

### 2.2 Funciones Empíricas  
Combinan términos con coeficientes ajustados a datos experimentales:
$$\Delta G = \Delta G_0 + \Delta G_{rot} + \Delta G_{HB} + \Delta G_{hydrophob} + \Delta G_{vdW}$$

### 2.3 Funciones de Conocimiento (Knowledge-based)
Derivadas de distribuciones estadísticas en complejos del PDB.

In [ ]:
def potencial_lennard_jones(r, epsilon=0.1, sigma=3.5):
    """Potencial de Lennard-Jones (componente vdW del scoring)."""
    return 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)

def potencial_electrostatico(r, q1=1.0, q2=-1.0, epsilon_r=4.0):
    """Potencial electrostático de Coulomb."""
    k = 332.06  # kcal·Å/(mol·e²)
    return k * q1 * q2 / (epsilon_r * r)

# Visualizar los potenciales
r = np.linspace(2.5, 10, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Lennard-Jones
lj = potencial_lennard_jones(r)
axes[0].plot(r, lj, 'b-', linewidth=2, label='LJ Total')
axes[0].plot(r, 4*0.1*(3.5/r)**12, 'r--', alpha=0.6, label='Repulsión (r⁻¹²)')
axes[0].plot(r, -4*0.1*(3.5/r)**6, 'g--', alpha=0.6, label='Atracción (-r⁻⁶)')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylim(-0.15, 0.5)
axes[0].set_xlabel('Distancia (Å)')
axes[0].set_ylabel('E (kcal/mol)')
axes[0].set_title('Potencial Lennard-Jones (vdW)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Electrostático
for q2, color, label in [(−1, 'red', 'Carga opuesta (−)'), 
                          (1, 'blue', 'Carga igual (+)')]:
    axes[1].plot(r, potencial_electrostatico(r, q2=q2), 
                color=color, linewidth=2, label=label)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Distancia (Å)')
axes[1].set_ylabel('E (kcal/mol)')
axes[1].set_title('Potencial Electrostático')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Scoring total simplificado
score_total = (potencial_lennard_jones(r) + 
               0.3 * potencial_electrostatico(r, q2=-0.5))
axes[2].plot(r, score_total, 'purple', linewidth=2)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Distancia (Å)')
axes[2].set_ylabel('Score (kcal/mol)')
axes[2].set_title('Score Total Simplificado')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Algoritmos de Búsqueda (Sampling)

El docking debe explorar el **espacio conformacional** del ligando para encontrar la pose de mínima energía. Este problema es de alta dimensionalidad:

- 3 grados de traslación (x, y, z)
- 3 grados de rotación (α, β, γ)
- N grados de libertad torsionales (enlaces rotativos del ligando)

### Principales algoritmos

| Algoritmo | Descripción | Usado en |
|-----------|-------------|---------|
| **Genético** | Evolución de poblaciones de poses | AutoDock, GOLD |
| **Monte Carlo** | Movimientos aleatorios + criterio de aceptación | AutoDock, ICM |
| **Gradiente** | Descenso por el gradiente de energía | GLIDE, MOE |
| **Fragmentación** | Ancla + crecimiento del ligando | FlexX, DOCK |
| **Grid** | Búsqueda exhaustiva en grilla discreta | AutoDock clásico |

In [ ]:
def simular_busqueda_monte_carlo(n_pasos=500, T=300, semilla=42):
    """
    Simulación simplificada de búsqueda Monte Carlo 2D.
    Representa la exploración del espacio conformacional.
    """
    np.random.seed(semilla)
    R = 1.987e-3  # kcal/(mol·K)
    
    def energia(x, y):
        """Superficie de energía 2D con múltiples mínimos (representa la PES)."""
        return (np.sin(x) * np.cos(y) + 
                0.5 * np.sin(2*x) * np.cos(3*y) + 
                0.3 * np.cos(x) * np.sin(2*y))
    
    # Estado inicial
    x, y = np.random.uniform(-np.pi, np.pi, 2)
    E_actual = energia(x, y)
    
    trayectoria_x = [x]
    trayectoria_y = [y]
    energias = [E_actual]
    aceptaciones = 0
    
    for paso in range(n_pasos):
        # Proponer movimiento aleatorio
        dx, dy = np.random.normal(0, 0.3, 2)
        x_nuevo = np.clip(x + dx, -np.pi, np.pi)
        y_nuevo = np.clip(y + dy, -np.pi, np.pi)
        E_nuevo = energia(x_nuevo, y_nuevo)
        
        # Criterio de Metropolis
        delta_E = E_nuevo - E_actual
        if delta_E < 0 or np.random.random() < np.exp(-delta_E / (R * T)):
            x, y = x_nuevo, y_nuevo
            E_actual = E_nuevo
            aceptaciones += 1
        
        trayectoria_x.append(x)
        trayectoria_y.append(y)
        energias.append(E_actual)
    
    return trayectoria_x, trayectoria_y, energias, aceptaciones

# Ejecutar simulación
traj_x, traj_y, energias, aceptaciones = simular_busqueda_monte_carlo(n_pasos=500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Superficie de energía + trayectoria
xx, yy = np.meshgrid(np.linspace(-np.pi, np.pi, 100),
                      np.linspace(-np.pi, np.pi, 100))
zz = (np.sin(xx)*np.cos(yy) + 0.5*np.sin(2*xx)*np.cos(3*yy) + 
      0.3*np.cos(xx)*np.sin(2*yy))

contour = axes[0].contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.8)
axes[0].plot(traj_x, traj_y, 'w-', alpha=0.4, linewidth=0.5)
axes[0].scatter(traj_x[0], traj_y[0], color='lime', s=100, zorder=5, 
               label=f'Inicio ({traj_x[0]:.2f},{traj_y[0]:.2f})')
axes[0].scatter(traj_x[-1], traj_y[-1], color='red', s=100, zorder=5, 
               marker='*', label=f'Final ({traj_x[-1]:.2f},{traj_y[-1]:.2f})')
plt.colorbar(contour, ax=axes[0], label='Energía (kcal/mol)')
axes[0].set_xlabel('Torsión 1 (rad)')
axes[0].set_ylabel('Torsión 2 (rad)')
axes[0].set_title(f'Exploración Monte Carlo\n(Aceptación: {aceptaciones/500*100:.1f}%)')
axes[0].legend(fontsize=8)

# Evolución de la energía
axes[1].plot(energias, color='steelblue', linewidth=1)
axes[1].axhline(min(energias), color='red', linestyle='--', 
               label=f'Mínimo: {min(energias):.3f}')
axes[1].set_xlabel('Paso MC')
axes[1].set_ylabel('Energía (kcal/mol)')
axes[1].set_title('Convergencia Energética')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"\nMínimo encontrado: {min(energias):.4f} kcal/mol en paso {np.argmin(energias)}")

## 4. AutoDock Vina: El Estándar del Área

**AutoDock Vina** (Trott & Olson, 2010) es el programa de docking más utilizado en investigación. Sus características:

- **Función de scoring**: Empírica + Gauss
- **Algoritmo**: Gradiente iterated local search + Monte Carlo
- **Flexibilidad**: Ligando flexible, receptor puede tener cadenas laterales flexibles
- **Grid box**: Define el espacio de búsqueda

### Parámetros clave de AutoDock Vina

```
receptor = proteina_preparada.pdbqt
ligand   = ligando.pdbqt
center_x = X    # Centro del sitio activo
center_y = Y
center_z = Z
size_x   = 20   # Tamaño de la caja (Å)
size_y   = 20
size_z   = 20
exhaustiveness = 8   # Mayor = más exhaustivo (más lento)
num_modes      = 9   # Número de poses a generar
```

### Flujo de trabajo completo con Vina

```
Receptor (PDB) → pdbqt         Ligando (SMILES/SDF) → pdbqt
                    ↓                                     ↓
              preparacion                          preparacion
                    ↓                                     ↓
                    └──────────── AutoDock Vina ──────────┘
                                       ↓
                               poses.pdbqt + scores
                                       ↓
                               análisis e interpretación
```

In [ ]:
def visualizar_grid_box(centro=(0, 0, 0), tamano=(20, 20, 20)):
    """
    Visualiza esquemáticamente el grid box de docking.
    Muestra el cubo que define el espacio de búsqueda.
    """
    from mpl_toolkits.mplot3d import Axes3D
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    
    cx, cy, cz = centro
    sx, sy, sz = tamano[0]/2, tamano[1]/2, tamano[2]/2
    
    # Vértices del cubo
    vertices = np.array([
        [cx-sx, cy-sy, cz-sz], [cx+sx, cy-sy, cz-sz],
        [cx+sx, cy+sy, cz-sz], [cx-sx, cy+sy, cz-sz],
        [cx-sx, cy-sy, cz+sz], [cx+sx, cy-sy, cz+sz],
        [cx+sx, cy+sy, cz+sz], [cx-sx, cy+sy, cz+sz]
    ])
    
    # Caras del cubo
    caras = [[vertices[j] for j in [0,1,2,3]],
             [vertices[j] for j in [4,5,6,7]],
             [vertices[j] for j in [0,1,5,4]],
             [vertices[j] for j in [2,3,7,6]],
             [vertices[j] for j in [0,3,7,4]],
             [vertices[j] for j in [1,2,6,5]]]
    
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # Dibujar cubo semitransparente
    poly = Poly3DCollection(caras, alpha=0.15, facecolor='cyan', edgecolor='blue')
    ax.add_collection3d(poly)
    
    # Centro de la caja
    ax.scatter(*centro, color='red', s=100, zorder=5, label='Centro del sitio activo')
    
    # Puntos de grilla
    xs = np.linspace(cx-sx, cx+sx, 5)
    ys = np.linspace(cy-sy, cy+sy, 5)
    zs = np.linspace(cz-sz, cz+sz, 5)
    gx, gy, gz = np.meshgrid(xs, ys, zs)
    ax.scatter(gx.ravel(), gy.ravel(), gz.ravel(), 
              color='gray', s=3, alpha=0.3, label='Puntos de grilla')
    
    ax.set_xlim(cx-sx-2, cx+sx+2)
    ax.set_ylim(cy-sy-2, cy+sy+2)
    ax.set_zlim(cz-sz-2, cz+sz+2)
    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    ax.set_title(f'Grid Box de Docking\nCentro: {centro}, Tamaño: {tamano} Å')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    
    print(f"Volumen del grid box: {tamano[0] * tamano[1] * tamano[2]:,} Å³")
    print(f"Consejo: El sitio activo debe estar completamente dentro de la caja")

# Visualizar grid box típico de docking
visualizar_grid_box(centro=(25.0, 45.0, 12.0), tamano=(22, 22, 22))

## 5. Interpretación de Resultados de Docking

### Energía de unión (Binding Affinity)

AutoDock Vina reporta la energía en kcal/mol. Valores de referencia:

| Score (kcal/mol) | Interpretación |
|-----------------|---------------|
| > -5 | Afinidad débil |
| -5 a -8 | Afinidad moderada |
| -8 a -10 | Buena afinidad |
| < -10 | Muy alta afinidad |

### RMSD entre poses
El RMSD mide la similitud geométrica entre dos poses:
$$RMSD = \sqrt{\frac{1}{N}\sum_{i=1}^{N}|r_i^{pose_1} - r_i^{pose_2}|^2}$$

- RMSD < 2 Å → Poses similares (mismo cluster)
- RMSD > 2 Å → Poses distintas

In [ ]:
def simular_resultados_docking(n_poses=9, semilla=42):
    """Simula resultados típicos de AutoDock Vina para demostración."""
    np.random.seed(semilla)
    
    scores = np.sort(np.random.normal(-8.0, 1.5, n_poses))
    rmsd_lb = np.zeros(n_poses)  # lb = lower bound
    rmsd_ub = np.zeros(n_poses)  # ub = upper bound
    
    for i in range(1, n_poses):
        rmsd_lb[i] = np.random.uniform(1.0, 8.0)
        rmsd_ub[i] = rmsd_lb[i] + np.random.uniform(0.5, 3.0)
    
    # Tabla de resultados (formato similar a Vina)
    print("=" * 55)
    print("  Modo  | Afinidad  | rmsd l.b. | rmsd u.b.")
    print("        | (kcal/mol)|    (Å)    |    (Å)   ")
    print("-" * 55)
    for i, (score, rl, ru) in enumerate(zip(scores, rmsd_lb, rmsd_ub), 1):
        marca = " ← MEJOR" if i == 1 else ""
        print(f"   {i:2d}   | {score:9.1f} | {rl:9.3f} | {ru:9.3f}{marca}")
    print("=" * 55)
    
    return scores, rmsd_lb, rmsd_ub

def graficar_resultados_docking(scores, rmsd_lb):
    """Grafica el análisis de poses de docking."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Scores por pose
    poses = np.arange(1, len(scores)+1)
    colores = ['gold' if i == 0 else 'steelblue' for i in range(len(scores))]
    axes[0].bar(poses, scores, color=colores, edgecolor='white')
    axes[0].set_xlabel('Modo (Pose)')
    axes[0].set_ylabel('Afinidad (kcal/mol)')
    axes[0].set_title('Energías de Docking por Pose')
    axes[0].set_xticks(poses)
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].text(1, scores[0]+0.1, 'Mejor pose', ha='center', fontsize=8, color='orange')
    
    # Score vs RMSD
    axes[1].scatter(rmsd_lb[1:], scores[1:], color='steelblue', 
                   s=80, alpha=0.8, label='Poses 2-9')
    axes[1].scatter(rmsd_lb[0], scores[0], color='gold', s=150, 
                   marker='*', zorder=5, label='Pose 1 (referencia)')
    axes[1].axvline(2.0, color='red', linestyle='--', alpha=0.6, label='RMSD = 2 Å')
    axes[1].set_xlabel('RMSD lb (Å)')
    axes[1].set_ylabel('Afinidad (kcal/mol)')
    axes[1].set_title('Score vs RMSD\n(agrupamiento de poses)')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

scores, rmsd_lb, rmsd_ub = simular_resultados_docking()
graficar_resultados_docking(scores, rmsd_lb)

## 6. Comparación de Programas de Docking

### Herramientas disponibles

| Software | Licencia | Velocidad | Precisión | Uso principal |
|---------|----------|-----------|-----------|--------------|
| **AutoDock Vina** | Gratuito | ⚡⚡⚡ | ⭐⭐⭐ | Investigación general |
| **AutoDock 4** | Gratuito | ⚡⚡ | ⭐⭐⭐ | Docking clásico |
| **GLIDE (Schrödinger)** | Comercial | ⚡⚡ | ⭐⭐⭐⭐⭐ | Industria farmacéutica |
| **GOLD (CCDC)** | Comercial | ⚡⚡ | ⭐⭐⭐⭐ | Predicción precisa |
| **DOCK 6** | Académico | ⚡⚡ | ⭐⭐⭐ | Validación |
| **rDock** | Gratuito | ⚡⚡⚡ | ⭐⭐⭐ | Virtual screening |
| **Smina** | Gratuito | ⚡⚡⚡ | ⭐⭐⭐ | Fork de Vina |

### Limitaciones del docking

- ❌ No captura efectos de solvatación completos
- ❌ Ignora la flexibilidad del receptor (generalmente)
- ❌ Las funciones de scoring no son perfectas
- ❌ Puede generar poses incorrectas (falsos positivos)
- ✅ Complementar siempre con validación experimental

## 7. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Usando la función `kd_to_dg`, calcula la $\Delta G$ para los siguientes valores de Kd:
- Ibuprofeno en COX-2: Kd ≈ 2.5 μM
- Aspirina en COX-1: Kd ≈ 0.3 μM
- Celecoxib en COX-2: Kd ≈ 40 nM

### Ejercicio 2 (Intermedio)
Modifica la función `simular_busqueda_monte_carlo` para:
1. Ejecutar con temperaturas T = 100K, 300K, 1000K
2. Comparar las tasas de aceptación y los mínimos encontrados
3. ¿Qué temperatura converge al mínimo global con más consistencia?

### Ejercicio 3 (Avanzado)
Investiga el concepto de **docking con receptor flexible** (Induced Fit Docking):
1. ¿Qué residuos se hacen flexibles típicamente?
2. ¿Cuál es el costo computacional adicional?
3. ¿Cuándo es necesario usar receptor flexible vs rígido?

In [ ]:
# Ejercicio 1: Calcular ΔG para fármacos antiinflamatorios
farmacos = {
    'Ibuprofeno (COX-2)': 2.5e-6,
    'Aspirina (COX-1)':   0.3e-6,
    'Celecoxib (COX-2)':  40e-9,
}

print("Fármaco                  | Kd           | ΔG (kcal/mol)")
print("-" * 55)
for nombre, kd in farmacos.items():
    dg = kd_to_dg(kd)
    print(f"{nombre:25s}| {kd*1e6:6.1f} μM  |  {dg:8.2f}")

# Tu código para ejercicios 2 y 3 aquí:


## 8. Referencias

1. Trott, O. & Olson, A.J. (2010). AutoDock Vina: improving the speed and accuracy of docking. *J. Comput. Chem.*, 31, 455-461.
2. Morris, G.M. et al. (2009). AutoDock4 and AutoDockTools4: Automated docking with selective receptor flexibility. *J. Comput. Chem.*, 30, 2785-2791.
3. Friesner, R.A. et al. (2004). Glide: A New Approach for Rapid, Accurate Docking and Scoring. *J. Med. Chem.*, 47, 1739-1749.
4. Leach, A.R. (2001). *Molecular Modelling: Principles and Applications*, 2nd ed. Pearson.
5. Shoichet, B.K. (2004). Virtual screening of chemical libraries. *Nature*, 432, 862-865.

---

## 📚 Recursos Adicionales

### Software
- [AutoDock Vina](http://vina.scripps.edu/) — Docking molecular gratuito
- [AutoDockTools (ADT)](http://mgltools.scripps.edu/) — GUI para AutoDock
- [UCSF DOCK](http://dock.compbio.ucsf.edu/) — Programa clásico de docking
- [Smina](https://github.com/mwojcikowski/smina) — Fork mejorado de Vina

### Tutoriales
- [AutoDock Vina Tutorial](http://vina.scripps.edu/tutorial.html)
- [PyRx Tutorial](https://pyrx.sourceforge.io/)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Explicar el concepto de docking molecular y sus hipótesis fundamentales
- ✅ Describir los componentes energéticos que determinan la afinidad de unión
- ✅ Diferenciar funciones de scoring (fuerza, empíricas, conocimiento)
- ✅ Entender el algoritmo Monte Carlo de búsqueda conformacional
- ✅ Configurar los parámetros del grid box de AutoDock Vina
- ✅ Interpretar los resultados de docking (scores y RMSD)

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 4.5: Fundamentos de Docking Molecular**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_4.4-Preparación_de_Proteínas-blue.svg)](04_preparacion_proteinas.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_4.6_➡️-Docking_Ligando_Proteína-green.svg)](06_docking_ligando_proteina.ipynb)

---

📚 **[Volver al Módulo 4](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>